In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import ipywidgets as widgets
from datetime import datetime
from help_functions import *
import warnings
warnings.filterwarnings('ignore')


In [ ]:
def load_data(folder_path="data_client"):
    """
    Load all relevant data files and prepare them for display
    """
    # Load the catalogue
    catalogue = pd.read_csv('catalogue.csv')
    
    # Get list of CSV files in the data folder
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    
    data_dict = {}
    for file in csv_files:
        # Extract the base name without extension
        base_name = os.path.splitext(file)[0]
        # Read the CSV file
        df = pd.read_csv(os.path.join(folder_path, file))
        # Convert date columns if they exist
        date_columns = [col for col in df.columns if 'date' in col.lower()]
        for date_col in date_columns:
            df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        data_dict[base_name] = df
    
    return data_dict, catalogue

# Load the data
data_dict, catalogue = load_data()

# Print available tables
print("Available tables:")
for table_name in data_dict.keys():
    print(f"- {table_name}")


In [ ]:
# Create widgets for table selection and filtering
table_selector = widgets.Dropdown(
    options=['Select Table'] + list(data_dict.keys()),
    value='Select Table',
    description='Table:',
    style={'description_width': 'initial'}
)

# Date range selector
date_range = widgets.DateRangeSlider(
    value=[datetime(2024, 1, 1), datetime(2024, 12, 31)],
    min=datetime(2024, 1, 1),
    max=datetime(2024, 12, 31),
    step=timedelta(days=1),
    description='Date Range:'
)

# Number of rows to display
rows_display = widgets.IntSlider(
    value=10,
    min=5,
    max=100,
    step=5,
    description='Rows:',
    style={'description_width': 'initial'}
)

# Search/filter text input
search_text = widgets.Text(
    value='',
    placeholder='Enter search terms...',
    description='Search:',
    style={'description_width': 'initial'}
)

# Column selector (will be updated based on selected table)
column_selector = widgets.SelectMultiple(
    options=[],
    value=[],
    description='Columns:',
    style={'description_width': 'initial'}
)

# Create layout
controls = widgets.VBox([
    table_selector,
    date_range,
    rows_display,
    search_text,
    column_selector
])

# Output widget for displaying the table
output = widgets.Output()


In [ ]:
def update_table(*args):
    with output:
        output.clear_output()
        
        if table_selector.value == 'Select Table':
            print("Please select a table to view")
            return
            
        # Get the selected table
        df = data_dict[table_selector.value].copy()
        
        # Update column selector options if table changed
        if args and args[0]['name'] == 'value' and args[0]['owner'] == table_selector:
            column_selector.options = list(df.columns)
            column_selector.value = list(df.columns)[:5]  # Select first 5 columns by default
        
        # Apply date filter if date columns exist
        date_columns = [col for col in df.columns if 'date' in col.lower()]
        if date_columns:
            for date_col in date_columns:
                mask = (df[date_col] >= pd.Timestamp(date_range.value[0])) & \
                      (df[date_col] <= pd.Timestamp(date_range.value[1]))
                df = df[mask]
        
        # Apply text search filter
        if search_text.value:
            search_mask = pd.Series(False, index=df.index)
            for col in df.columns:
                search_mask |= df[col].astype(str).str.contains(search_text.value, case=False, na=False)
            df = df[search_mask]
        
        # Select columns
        if column_selector.value:
            df = df[list(column_selector.value)]
        
        # Display the filtered table
        if len(df) > 0:
            display(HTML(df.head(rows_display.value).to_html(
                classes='table table-striped table-bordered',
                float_format=lambda x: '{:.2f}'.format(x) if isinstance(x, (float, np.floating)) else x
            )))
            print(f"\nTotal rows in filtered data: {len(df)}")
        else:
            print("No data matches the current filters")

# Register the update function with the widgets
table_selector.observe(update_table, names='value')
date_range.observe(update_table, names='value')
rows_display.observe(update_table, names='value')
search_text.observe(update_table, names='value')
column_selector.observe(update_table, names='value')

# Display the interface
display(controls, output)


In [ ]:
from IPython.display import HTML

# Add custom CSS styling
display(HTML("""
<style>
    .widget-dropdown {
        width: 300px !important;
    }
    .widget-text {
        width: 300px !important;
    }
    .widget-select-multiple {
        width: 300px !important;
        height: 200px !important;
    }
    .table {
        font-size: 12px;
        margin-top: 20px;
    }
    .table th {
        background-color: #f8f9fa;
        position: sticky;
        top: 0;
    }
    .widget-html-style {
        max-height: 600px;
        overflow-y: auto;
    }
</style>
"""))
